# Lab 14 · Pierwszy model na danych projektu

Przed wykonaniem tego laboratorium przeczytaj wykłady [C12.1 „Oblicz prognozę z jednej cechy”](../lectures/c12-linear.html), [C12.2 „Wytrenuj pierwszy model i zachowaj test”](../lectures/c12-fit.html) i [C12.3 „Dopasuj skalowanie tylko na treningu”](../lectures/c12-pipeline.html).

Pamiętaj, że do tego labu potrzebne jest środowisko z labu 9: projekt practice z bibliotekami sekcji C. Uruchom Jupyter tak samo jak tam.

Pod opisem każdego zadania znajduje się komórka kodu do uzupełnienia, a pod nią komórka sprawdzenia. Jeżeli zadanie nie jest jeszcze rozwiązane, sprawdzenie zatrzyma się i wyświetli czerwony komunikat. To normalne: komunikat mówi, co trzeba poprawić.

Najpierw przewidź wynik, potem uruchom kod. Podpowiedzi otwieraj pojedynczo.

In [ ]:
from pathlib import Path
import sys
assert sys.version_info[:2] == (3, 12), "Wybierz kernel Pythona 3.12 z głównego README."
lab_dir = Path.cwd()
assert (lab_dir / "lab-14-pierwszy-model.ipynb").is_file(), "Uruchom Jupyter z folderu labs."
print("Folder:", lab_dir)

In [ ]:
try:
    import numpy as np
    import pandas as pd
except ImportError as error:
    raise AssertionError("Brakuje bibliotek sekcji C. Dodaj je w practice i uruchom Jupyter przez --project ../practice, zgodnie z głównym README.") from error
print("pandas", pd.__version__, "| numpy", np.__version__)
import json
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

## C12.1 · Prognoza liniowa na kartce

Po wykładzie C12.1 model ma wagę `w` i wyraz wolny `b`. W `predictions` zapisz listę prognoz dla trzech wartości cechy z `features`, licząc `w * x + b`. W `parameters` zapisz listę nazw tych wielkości, które model dobiera podczas fit. Najpierw policz prognozy ręcznie.

In [ ]:
w = 0.8
b = 3.0
features = [0.0, 5.0, 10.0]
predictions = None
parameters = None

In [ ]:
assert isinstance(predictions, list) and len(predictions) == 3, "C12.1: jedna prognoza na każdą wartość cechy."
assert all(abs(got - wanted) < 1e-9 for got, wanted in zip(predictions, [3.0, 7.0, 11.0])), "C12.1: prognoza to w * x + b; wyraz wolny dodajesz w każdym wierszu."
assert isinstance(parameters, list) and set(parameters) == {"w", "b"}, "C12.1: parametry dobierane podczas fit to waga i wyraz wolny; cecha jest wejściem, nie parametrem."
print("C12.1: trzy prognozy i dwa parametry są rozróżnione.")

<details><summary>Podpowiedź 1</summary>

Dla cechy 0 prognoza to sam wyraz wolny.

</details>

<details><summary>Podpowiedź 2</summary>

Comprehension po features daje listę trzech liczb. Parametrów jest dwa.

</details>

## Dane projektu

Gotowa komórka wczytuje `data/project-development.csv`: 49 dni syntetycznej sprzedaży jednego produktu, bez danych klientów. Pierwsze 42 dni to trening, kolejne 7 walidacja; `cutoff` to pierwszy dzień walidacji. Osobny plik `project-holdout.csv` zostaje nieotwarty do F21.5. Jeżeli go już obejrzano, zapisz to ograniczenie przy wynikach.

Protokół projektu: prognoza jednodniowa aktualizowana codziennie. Dla każdego dnia walidacji znasz rzeczywistą sprzedaż poprzedniego dnia, bez ponownego fit na ocenianych dniach.

In [ ]:
sales = pd.read_csv(lab_dir / "data" / "project-development.csv", parse_dates=["day"])
assert sales.shape == (49, 2), "Oczekiwano 49 dni i dwóch kolumn."
cutoff = pd.Timestamp("2024-02-12")
print(sales.dtypes)
print("Od", sales["day"].min().date(), "do", sales["day"].max().date(), "| pierwszy dzień walidacji:", cutoff.date())
print(sales.head(3))

## C12.2 · Cecha, wiersze z pełną cechą i dwa okresy

Po wykładzie C12.2 w `with_lag` dodaj do `sales` kolumnę `lag1` ze sprzedażą poprzedniego dnia, nie zmieniając `sales`. W `rows` zostaw tylko wiersze ze znaną cechą; pierwszy dzień jawnie odpada. W `train` zapisz wiersze z dniem przed `cutoff`, a w `valid` od `cutoff`. Przewidź liczbę wierszy w obu tabelach, zanim je policzysz.

In [ ]:
with_lag = None
rows = None
train = None
valid = None
# assign i shift(1), maska notna, dwie maski po kolumnie day względem cutoff.

In [ ]:
assert isinstance(train, pd.DataFrame) and isinstance(valid, pd.DataFrame), "C12.2: train i valid mają być tabelami."
assert "lag1" in train.columns and "lag1" in valid.columns, "C12.2: obie tabele potrzebują kolumny lag1."
assert train["lag1"].notna().all() and valid["lag1"].notna().all(), "C12.2: usuń jawnie wiersz bez cechy przed podziałem."
assert len(train) == 41 and len(valid) == 7, "C12.2: 41 dni treningu od 2 stycznia i 7 dni walidacji."
assert train["day"].max() == pd.Timestamp("2024-02-11") and valid["day"].min() == cutoff and valid["day"].max() == pd.Timestamp("2024-02-18"), "C12.2: trening kończy się dzień przed cutoff, walidacja zaczyna się w cutoff."
assert valid["lag1"].tolist() == [17.15, 17.3, 22.45, 19.6, 21.75, 24.9, 24.05], "C12.2: lag1 w dniu walidacji to rzeczywista sprzedaż poprzedniego dnia."
assert sales.shape == (49, 2), "C12.2: sales pozostaje bez zmian; nową kolumnę dodaje assign."
print("C12.2: granice dat i liczby wierszy są zapisane w tabelach.")

<details><summary>Podpowiedź 1</summary>

shift(1) na kolumnie sale daje sprzedaż poprzedniego dnia. Pierwszy wiersz nie ma poprzednika.

</details>

<details><summary>Podpowiedź 2</summary>

Maska notna wybiera wiersze z cechą. Kolumnę dat porównasz z cutoff tak jak liczby.

</details>

## C12.2 · Baseline na tych samych dniach

W `baseline_mae` zapisz MAE prognozy „tak samo jak wczoraj” na dniach walidacji: prognozą każdego dnia jest jego `lag1`. Policz jeden błąd ręcznie, zanim uruchomisz kod. Baseline nie ma etapu fit.

In [ ]:
baseline_mae = None

In [ ]:
import math
assert baseline_mae is not None and math.isclose(float(baseline_mae), 2.735714285714285, rel_tol=1e-6), "C12.2: porównaj sale z lag1 wyłącznie na valid; siedem błędów, jedna średnia."
print("C12.2: baseline MAE na walidacji:", round(float(baseline_mae), 4), "[szt.]")

<details><summary>Podpowiedź 1</summary>

Błąd dnia to różnica sale i lag1 w tym samym wierszu valid.

</details>

<details><summary>Podpowiedź 2</summary>

Wartość bezwzględna różnicy kolumn, a potem mean. Jednostka wyniku to sztuki.

</details>

## C12.2 · Regresja dopasowana tylko na treningu

W `model` zapisz `LinearRegression` dopasowaną metodą fit na `train[["lag1"]]` i `train["sale"]`. W `model_mae` zapisz MAE jej prognoz na `valid`, na tych samych siedmiu dniach co baseline. Nie dodawaj walidacji do fit. Zapisz oba wyniki obok siebie, także jeśli baseline wypadnie lepiej.

In [ ]:
model = None
model_mae = None

In [ ]:
import math
assert isinstance(model, LinearRegression) and hasattr(model, "coef_"), "C12.2: model to LinearRegression po wywołaniu fit."
assert math.isclose(float(model.coef_[0]), 0.16524662720106117, rel_tol=1e-6), "C12.2: dopasuj model wyłącznie na train; inna waga oznacza inne wiersze treningu."
assert model_mae is not None and math.isclose(float(model_mae), 3.574861056584409, rel_tol=1e-6), "C12.2: MAE licz z predict na valid[[\"lag1\"]] wobec valid[\"sale\"]."
print("C12.2: model MAE na walidacji:", round(float(model_mae), 4), "[szt.] | baseline:", round(float(baseline_mae), 4), "[szt.]")

<details><summary>Podpowiedź 1</summary>

fit przyjmuje tabelę cech z podwójnymi nawiasami i kolumnę celu.

</details>

<details><summary>Podpowiedź 2</summary>

predict na valid[["lag1"]] daje tablicę siedmiu prognoz. MAE licz tak jak dla baseline.

</details>

## C12.3 · Scaler i Ridge uczą się tylko na treningu

Po wykładzie C12.3 dla każdej wartości z `alphas` zbuduj `make_pipeline(StandardScaler(), Ridge(alpha=...))`, dopasuj na treningu i zapisz w słowniku `pipelines` pod kluczem alpha. W `ridge_mae` zapisz MAE każdego pipeline na `valid`. Scaler ma poznać średnią wyłącznie z treningu; sprawdzenie odczyta ją z `named_steps["standardscaler"].mean_`.

In [ ]:
alphas = [0.1, 10.0]
pipelines = {}
ridge_mae = {}
# Pętla po alphas: pipeline, fit na train, MAE na valid.

In [ ]:
import math
from sklearn.pipeline import Pipeline
assert set(pipelines) == set(alphas) and set(ridge_mae) == set(alphas), "C12.3: po jednym pipeline i jednym MAE dla każdej wartości alpha."
for alpha in alphas:
    assert isinstance(pipelines[alpha], Pipeline), "C12.3: użyj make_pipeline ze StandardScaler i Ridge."
    scaler_mean = float(pipelines[alpha].named_steps["standardscaler"].mean_[0])
    assert math.isclose(scaler_mean, 16.975609756097562, rel_tol=1e-6), "C12.3: scaler poznał średnią spoza treningu; fit wykonaj na train."
assert math.isclose(float(ridge_mae[0.1]), 3.5764905900310064, rel_tol=1e-6) and math.isclose(float(ridge_mae[10.0]), 3.7061822813984238, rel_tol=1e-6), "C12.3: MAE licz z predict na valid, dla obu alpha na tych samych wierszach."
print("C12.3: MAE Ridge na walidacji:", {alpha: round(float(value), 4) for alpha, value in ridge_mae.items()}, "[szt.]")

<details><summary>Podpowiedź 1</summary>

Pętla for alpha in alphas wypełnia oba słowniki; klucz to alpha.

</details>

<details><summary>Podpowiedź 2</summary>

Przykład wykładu C12.3 pokazuje make_pipeline i fit. MAE licz jak w C12.2, z predict na valid.

</details>

## C12.3 · Zapisz przebieg eksperymentu

Zapisz wyniki w pliku `results/c12-project.json`, obliczając pola z gotowych zmiennych, nie z pamięci. Słownik ma pola:

- `feature`: napis `lag1`; `protocol`: napis `daily-updated-one-day-ahead`,
- `train` i `valid`: słowniki z `start` i `end` jako daty w formacie `RRRR-MM-DD` oraz `rows` jako liczba wierszy,
- `baseline_mae` i `linear_mae`: liczby z notebooka,
- `ridge_mae`: słownik z kluczami `"0.1"` i `"10.0"`,
- `chosen`: nazwa wariantu z najniższym MAE na walidacji, jedna z `baseline`, `linear`, `ridge_0.1`, `ridge_10.0`,
- `holdout_opened`: False, albo True, jeśli plik holdoutu był już oglądany.

Ten zapis to dane eksperymentu, do których wrócisz w D14 i F21.5. Nie wymaga osobnego komentarza.

In [ ]:
result_dir = lab_dir / "results"
result_dir.mkdir(exist_ok=True)
report_path = result_dir / "c12-project.json"

In [ ]:
# Zbuduj słownik z pól opisanych wyżej i zapisz go przez json.dumps i write_text do report_path.

In [ ]:
import math
assert report_path.is_file(), "C12.3: zapisz plik results/c12-project.json."
report = json.loads(report_path.read_text(encoding="utf-8"))
assert report.get("feature") == "lag1" and report.get("protocol") == "daily-updated-one-day-ahead", "C12.3: zapisz cechę i protokół dokładnie w podanej formie."
assert report.get("train") == {"start": "2024-01-02", "end": "2024-02-11", "rows": 41}, "C12.3: granice i liczba wierszy treningu z tabeli train; daty jako RRRR-MM-DD."
assert report.get("valid") == {"start": "2024-02-12", "end": "2024-02-18", "rows": 7}, "C12.3: granice i liczba wierszy walidacji z tabeli valid."
assert math.isclose(report["baseline_mae"], float(baseline_mae), rel_tol=1e-6) and math.isclose(report["linear_mae"], float(model_mae), rel_tol=1e-6), "C12.3: wyniki w pliku mają zgadzać się ze zmiennymi notebooka."
assert set(report.get("ridge_mae", {})) == {"0.1", "10.0"} and all(math.isclose(report["ridge_mae"][str(alpha)], float(ridge_mae[alpha]), rel_tol=1e-6) for alpha in alphas), "C12.3: ridge_mae ma klucze \"0.1\" i \"10.0\" z wynikami z notebooka."
scores = {"baseline": float(baseline_mae), "linear": float(model_mae), "ridge_0.1": float(ridge_mae[0.1]), "ridge_10.0": float(ridge_mae[10.0])}
assert report.get("chosen") == min(scores, key=scores.get), "C12.3: chosen wskazuje wariant z najniższym MAE na walidacji; przewaga prostszego wariantu też jest wynikiem."
assert isinstance(report.get("holdout_opened"), bool), "C12.3: holdout_opened to True albo False."
print("C12.3: przebieg zapisany. Wybór na walidacji:", report["chosen"])

<details><summary>Podpowiedź 1</summary>

Datę z kolumny day zamienisz na tekst przez str(train["day"].min().date()).

</details>

<details><summary>Podpowiedź 2</summary>

Porównaj cztery wyniki MAE i wybierz najmniejszy. json.dumps zapisze słownik, a liczby float i bool są dozwolone.

</details>

## Sprawdź rezultat

Sprawdzenia przechodzą po restarcie kernela, a plik results/c12-project.json zawiera daty, cechę, protokół i cztery wyniki. Na tych danych prosty baseline może wygrać z regresją; to poprawny wynik eksperymentu, nie błąd. Do przemyślenia: dlaczego regresja dopasowana na wszystkich 48 wierszach dałaby lepszy wynik na walidacji i dlaczego nie wolno go tak liczyć? Czy pipeline uchroniłby przed użyciem jutrzejszej sprzedaży jako cechy? Plik holdout pozostaje zamknięty do F21.5.